In [1]:
import os
import gc
import pandas as pd
import pickle
import matplotlib.pyplot as plt

import anndata as ad
import numpy as np
import yaml
import sys
import h5py
import logging
import scanpy as sc
from os.path import join
import scipy.io as sio
import scipy.sparse as sps
from sklearn.cluster import KMeans
import gzip
from scipy.io import mmread
from pathlib import Path, PurePath
from sklearn.metrics import adjusted_rand_score


import warnings
def wrap_warn_plot(adata, basis, color, **kwargs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        sc.pl.embedding(adata, basis=basis, color=color, **kwargs)

def get_umap(ad, use_reps=[]):
    for use_rep in use_reps:
        umap_add_key = f'{use_rep}_umap'
        sc.pp.neighbors(ad, use_rep=use_rep, n_neighbors=15)
        sc.tl.umap(ad)
        ad.obsm[umap_add_key] = ad.obsm['X_umap']
    return ad

from sklearn.metrics import adjusted_rand_score

def split_ob(ads, ad_ref, ob='obs', key='emb2'):
    len_ads = [_.n_obs for _ in ads]
    if ob=='obsm':
        split_obsms = np.split(ad_ref.obsm[key], np.cumsum(len_ads[:-1]))
        for ad, v in zip(ads, split_obsms):
            ad.obsm[key] = v
    else:
        split_obs = np.split(ad_ref.obs[key].to_list(), np.cumsum(len_ads[:-1]))
        for ad, v in zip(ads, split_obs):
            ad.obs[key] = v


import json
import copy
from matplotlib.image import imread
def flip_coords(ads):
    for ad in ads:
        ad.obsm['spatial'] = -1 * ad.obsm['spatial']
        ad.obsm['spatial'] = ad.obsm['spatial'][:, ::-1]

def reorder(ad1, ad2):
    shared_barcodes = ad1.obs_names.intersection(ad2.obs_names)
    ad1 = ad1[shared_barcodes].copy()
    ad2 = ad2[shared_barcodes].copy()
    return ad1, ad2

def load_peak_expr(_dir):
    data = sio.mmread(join(_dir, 'data.mtx'))
    cname = pd.read_csv(join(_dir, 'barcode.csv'), index_col=0)['x'].to_list()
    feat = pd.read_csv(join(_dir, 'feat.csv'), index_col=0)['x'].to_list()
    ad = sc.AnnData(sps.csr_matrix(data.T))
    ad.obs_names = cname
    ad.var_names = feat
    return ad

/home/yanxh/anaconda3/envs/torch112/lib/python3.8/site-packages/setuptools_scm/_integration/setuptools.py:30: RuntimeWarning: 
ERROR: setuptools==59.5.0 is used in combination with setuptools_scm>=8.x

Your build configuration is incomplete and previously worked by accident!
setuptools_scm requires setuptools>=61

Suggested workaround if applicable:
 - migrating from the deprecated setup_requires mechanism to pep517/518
   and using a pyproject.toml to declare build dependencies
   which are reliably pre-installed before running the build tools

  warnings.warn(


In [2]:
def set_col2cat(ad, cols=[]):
    for col in cols:
        ad.obs[col] = ad.obs[col].astype('category')

def unify_colors(queries, color_key, ref_color_dict):
    for q in queries:
        q.obs[color_key] = q.obs[color_key].astype('category')
        q.uns[f'{color_key}_colors'] = [ref_color_dict[_] for _ in q.obs[color_key].cat.categories]
    return queries

def subset_ad(ad, subset_index):
    ad = ad[subset_index].copy()
    return ad

def set_spatial(ad):
    ad.obsm['spatial'] = ad.obs[['array_row', 'array_col']].values
    ad.obsm['spatial'] = ad.obsm['spatial'][:, ::-1]
    ad.obsm['spatial'][:, 1] = -1 * ad.obsm['spatial'][:, 1]
    return ad

In [3]:
from collections import Counter
def load_zu(_dir):
    zs = []

    for fi in sorted(os.listdir(_dir)):
        dfi = pd.read_csv(join(_dir, fi), header=None)
        zs.append(dfi.values)
    zs = np.vstack(zs)
    z, u = zs[:, :-2], zs[:, -2:]
    return z, u

def wrap_warn_plot(adata, basis, color, **kwargs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        sc.pl.embedding(adata, basis=basis, color=color, **kwargs)

def get_umap(ad, use_reps=[]):
    for use_rep in use_reps:
        umap_add_key = f'{use_rep}_umap'
        sc.pp.neighbors(ad, use_rep=use_rep, n_neighbors=15)
        sc.tl.umap(ad)
        ad.obsm[umap_add_key] = ad.obsm['X_umap']
    return ad

In [4]:
result_dir = './result'

ds = 'Mux-VisiumHD-HE'

In [5]:
data_dir = '../../../data/processed/Mux-VisiumHD-HE'

ad_mux_rna = sc.read_h5ad(join(data_dir, 'Mux/ad_rna.h5ad'))
ad_mux_atac = sc.read_h5ad(join(data_dir, 'Mux/ad_atac.h5ad'))

ad_vhd_rna = sc.read_h5ad(join(data_dir, 'VisiumHD/ad_rna.h5ad'))
ad_vhe_rna = sc.read_h5ad(join(data_dir, 'VisiumHE/ad_rna.h5ad')) 

In [6]:
ad_rna_all = sc.concat([ad_mux_rna, ad_vhd_rna, ad_vhe_rna])

sc.pp.highly_variable_genes(ad_rna_all, flavor='seurat_v3', n_top_genes=10000, batch_key=batch_key)
hvg_names = ad_rna_all.var.query('highly_variable').index.to_numpy()

# ac.pp.tfidf(ad_atac_all, scale_factor=1e4)
sc.pp.highly_variable_genes(ad_mux_atac, flavor='seurat_v3', n_top_genes=100000, batch_key=batch_key)
hvp_names = ad_mux_atac.var.query('highly_variable').index.to_numpy()

In [7]:
ad_mux_rna = ad_mux_rna[:, hvg_names].copy()
ad_vhd_rna = ad_vhd_rna[:, hvg_names].copy()
ad_vhe_rna = ad_vhe_rna[:, hvg_names].copy()

ad_mux_atac = ad_mux_atac[:, hvp_names].copy()

In [8]:
## filter feat names
filtered_atac_feats = [_ for _ in ad_mux_atac.var_names if _.startswith('chr')]
ad_mux_atac = ad_mux_atac[:, filtered_atac_feats].copy()

In [9]:
def split_list_byChr(input_list):
    atac_name_chunks = []
    for n in input_list:
        chr = n.split("-")[0]
        if len(atac_name_chunks) == 0 or chr!=atac_name_chunks[-1]:
            atac_name_chunks.append(chr)
    return atac_name_chunks

In [16]:
### save data
tmp_out_dir = f'./data/processed/{ds}'
feat_dir = join(tmp_out_dir, 'feat')
os.makedirs(feat_dir, exist_ok=True)

ad_mod_rna = ad_mux_rna
ad_mod_atac = ad_mux_atac

atac_feat_chunks = split_list_byChr(ad_mod_atac.var_names) # split the chr-? in order

atac_chr_count = Counter([_.split('-')[0] for _ in ad_mod_atac.var_names]) # count the frequency of each chr-?
df_feat_dims = pd.DataFrame(np.array([atac_chr_count[_] for _ in atac_feat_chunks]).reshape(-1, 1), columns=['atac'])
df_feat_dims['rna'] = ad_mod_rna.n_vars
df_feat_rna_names = pd.DataFrame(list(ad_mod_rna.var_names), columns=['x'])
df_feat_adt_names = pd.DataFrame(list(ad_mod_atac.var_names), columns=['x'])
df_feat_dims.to_csv(join(feat_dir, 'feat_dims.csv'))
df_feat_rna_names.to_csv(join(feat_dir, 'feat_names_rna.csv'))
df_feat_adt_names.to_csv(join(feat_dir, 'feat_names_atac.csv'))

In [13]:
# each subset
subsets = [[ad_mux_rna, ad_mux_atac], [ad_vhd_rna], [ad_vhe_rna]]
mods = [['rna', 'atac'], ['rna'], ['rna']]

for si in range(len(subsets)):
    for fname in ['mask', 'mat', 'vec']:
        os.makedirs(join(tmp_out_dir, f'subset_{si}/{fname}'), exist_ok=True)
    tmp_dir = join(tmp_out_dir, f'subset_{si}')
    for ad,mi in zip(subsets[si], mods[si]):
        mat = ad.X.A if sps.issparse(ad.X) else ad.X
        df_mat = pd.DataFrame(mat, index=ad.obs_names, columns=ad.var_names)
        df_mat.to_csv(join(tmp_dir, f'mat/{mi}.csv'))

        os.makedirs(join(tmp_dir, f'vec/{mi}'), exist_ok=True)
        for idx, mati in enumerate(mat):
            pd.DataFrame(mati.reshape(1, -1)).to_csv(join(tmp_dir, 'vec/{}/{:05d}.csv'.format(mi, idx)), header=None, index=None)
        pd.DataFrame(ad.obs_names, columns=['x']).to_csv(join(tmp_dir, 'cell_names.csv'))

        # save mask
        pd.DataFrame(np.ones(ad.n_vars, dtype='int')).to_csv(join(tmp_dir, f'mask/{mi}.csv'))

In [15]:
for e, ep in enumerate([2000], 1):
    training_command = f'CUDA_VISIBLE_DEVICES=1 python run.py --exp e{e} --task {ds} --epoch_num {ep}'

    run_command = 'CUDA_VISIBLE_DEVICES=1 python run.py --task {} --act predict_all_latent_bc --init_model sp_{:08d} --exp e{}'\
                                .format(ds, ep-1, e)
    print(training_command)
    print(run_command)

CUDA_VISIBLE_DEVICES=1 python run.py --exp e1 --task Mux-VisiumHD-HE --epoch_num 2000
CUDA_VISIBLE_DEVICES=1 python run.py --task Mux-VisiumHD-HE --act predict_all_latent_bc --init_model sp_00001999 --exp e1


In [ ]:
e, ep = 1, 2000
model_pt = 'sp_{:08d}'.format(ep-1)

tmp_dir = f'{result_dir}/{ds}/e{e}/default/predict/{model_pt}'
target_ads = [ad_mux_rna, ad_vhd_rna, ad_vhe_rna]
for i in range(len(target_ads)):
    z, u = load_zu(join(tmp_dir, f'subset_{i}/z/joint'))
    target_ads[i].obsm['z'] = z
    target_ads[i].obsm['u'] = u
    
ad_mosaic = sc.concat(target_ads)
# ad_mosaic = get_umap(ad_mosaic, ['z'])

# wrap_warn_plot(ad_mosaic, 'z_umap', ['src'])

## Vis

In [ ]:
anal_dir = '../../../results/embeddings/MIDAS/Mux-VisiumHD-HE'
os.makedirs(anal_dir, exist_ok=True)

df = pd.DataFrame(ad_mosaic.obsm['z'], index=ad_mosaic.obs_names)
df.to_csv(join(anal_dir, 'df_emb.csv'))